In [5]:
# Library essential
import numpy as np
import requests
from io import StringIO
import pickle
import csv
import joblib

# Library pre/processing kata (dalam bentuk string)
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [10]:
model_dnn = joblib.load("model_dnn.h5")
tfidf = joblib.load("tfidf.h5")

In [3]:
# 1. Case Folding
def CaseFolding(text):
    text = text.lower()
    return text

# 2. Removal Special Characters
def RemoveSpecialCharacter(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # remove mentions
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # remove hashtag
    text = re.sub(r'RT[\s]', '', text) # remove RT
    text = re.sub(r"http\S+", '', text) # remove link
    text = re.sub(r'[0-9]+', '', text) # remove numbers
    text = re.sub(r'[^\w\s]', '', text) # remove numbers
    text = text.replace('\n', ' ') # replace new line into space
    text = text.translate(str.maketrans('', '', string.punctuation)) # remove all punctuations
    text = text.strip(' ') # remove characters space from both left and right text
    return text

# 3. Stopword Removal
def RemoveStopwords(text):
    listStopwords = set(stopwords.words('english'))
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

# 4. Tokenizing
def TokenizeText(text):
    text = word_tokenize(text)
    return text

# 5. Lemmatization
def LemmatizeText(text):
    lemmatizer = WordNetLemmatizer()

    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word.lower()) for word in words]
    lemmatized_text = ' '.join(lemmatized_words)
    return lemmatized_text


# Tambahan dari Colab contoh
def ToSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

# Menggunakan dictionary slang bahasa Inggris dari GitHub
# Credits: https://github.com/sifei/Dictionary-for-Sentiment-Analysis
slangwords = dict()
response = requests.get('https://raw.githubusercontent.com/sifei/Dictionary-for-Sentiment-Analysis/refs/heads/master/slang/acrynom.csv')

if response.status_code == 200:
  reader = csv.reader(StringIO(response.text), delimiter=',')
  for rows in reader:
    slangwords[rows[0]] = rows[1]
else:
    print("Error mengambil dictionary slangword")

def FixSlangwords(text):
  words = text.split()
  fixed_words = []

  for word in words:
    if word.lower() in slangwords:
      fixed_words.append(slangwords[word.lower()])
    else:
      fixed_words.append(word)
  fixed_text = ' '.join(fixed_words)
  return fixed_text

# Credits: sebagian besar blok kode ini dimodifikasi dari versi asli Google Colab contoh penerapan NLP
# https://colab.research.google.com/drive/173RsZ-l3SAd2VZwKisXisxYbKKsp28Qd

In [20]:
def Prediksi_Sentimen(kalimat):
  # Preprocessing
  kalimat_baru_cleaned = (kalimat)
  kalimat_baru_casefolded = CaseFolding(kalimat_baru_cleaned)
  kalimat_baru_slangfixed = FixSlangwords(kalimat_baru_casefolded)
  kalimat_baru_tokenized = TokenizeText(kalimat_baru_slangfixed)
  kalimat_baru_filtered = RemoveStopwords(kalimat_baru_tokenized)
  kalimat_baru_final = ToSentence(kalimat_baru_filtered)

  X_kalimat_baru = tfidf.transform([kalimat_baru_final]).toarray()

  # Predict label kalimat
  prediksi_sentimen = model_dnn.predict(X_kalimat_baru)

  hasil_prediksi = np.argmax(prediksi_sentimen, axis=1)
  polaritas = ""
  if hasil_prediksi == 0:
    polaritas = "Negatif"
  elif hasil_prediksi == 1:
    polaritas = "Netral"
  elif hasil_prediksi == 2:
    polaritas = "Positif"

  print("Sentimen kalimat baru adalah", polaritas)

In [24]:
kalimat_baru = input("Masukkan kalimat baru: ")
Prediksi_Sentimen(kalimat_baru)

Masukkan kalimat baru: great game, i love it! so many cars, customization, and exciting events and updates regularly
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Sentimen kalimat baru adalah Positif


In [25]:
kalimat_baru = input("Masukkan kalimat baru: ")
Prediksi_Sentimen(kalimat_baru)

Masukkan kalimat baru: boring gameplay, not much thing to do, very repetitive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Sentimen kalimat baru adalah Negatif


In [26]:
kalimat_baru = input("Masukkan kalimat baru: ")
Prediksi_Sentimen(kalimat_baru)

Masukkan kalimat baru: no comment
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Sentimen kalimat baru adalah Netral
